# Phase 4 : Intelligence Artificielle Generative (LLM)
Dans ce notebook, nous comparons deux approches modernes de NLP :
1. **Zero-Shot Classification** (avec un modele Encoder type BERT/BART) : On classe sans entrainement.
2. **Few-Shot Learning** (avec un modele Decoder type GPT) : On donne des exemples a l'IA pour qu'elle complete la tache.

In [ ]:
from transformers import pipeline, set_seed
import pandas as pd
import os
import sys
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_yelp_sample

# Pour la reproductibilite des resultats GPT
set_seed(42)

## Etape 1 : Chargement d'un avis pour le test

In [ ]:
# On recupere un avis negatif aleatoire pour tester nos deux IA
df = load_yelp_sample('../data/raw/review.json', n_rows=1000)
bad_review = df[df['stars'] == 1].iloc[0]['text']
print(f'--- Texte a analyser ---')
print(f'{bad_review[:200]}...')

Chargement de 1000 lignes depuis ../data/raw/review.json...
--- Texte a analyser ---
I am a long term frequent customer of this establishment. I just went in to order take out (3 apps) and was told they're too busy to do it. Really? The place is maybe half full at best. Does your dick...


## Approche 1 : Zero-Shot Classification (Encoder)
Utilisation de `distilbart` (modele type BERT) qui comprend le sens des categories sans exemples.

In [ ]:
# On definit des categories arbitraires
classifier = pipeline('zero-shot-classification', model='valhalla/distilbart-mnli-12-3')
labels = ['food quality', 'service', 'price', 'ambiance']

res = classifier(bad_review, labels)

print(f'Resultat Zero-Shot : Le probleme vient de : {res["labels"][0].upper()}')
print(f'Indice de confiance : {res["scores"][0]:.2%}')

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Resultat Zero-Shot : Le probleme vient de : SERVICE
Indice de confiance : 43.93%


## Approche 2 : Few-Shot Learning (Decoder)
C'est la methode demandee pour les LLM generatifs (comme ChatGPT ou DeepSeek).
On utilise ici **GPT-2** (un Decoder pur) et on lui fournit un prompt contenant des exemples (Few-Shot) pour qu'il comprenne la logique.

In [ ]:
from transformers import pipeline
import torch

# On choisit le modèle DeepSeek-R1 version "Distill" (1.5 milliards de paramètres).
# C'est un bon compromis : il est capable de raisonner comme les gros modèles
# mais reste assez léger pour tourner sur un ordinateur personnel.
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

print(f"Chargement du modèle {model_name}...")

# Initialisation du pipeline de génération.
# On force l'utilisation du CPU (device=-1) pour éviter les erreurs de compatibilité
# si l'ordinateur n'a pas de carte graphique Nvidia (CUDA).
generator = pipeline(
    "text-generation",
    model=model_name,
    torch_dtype=torch.float32, 
    device=-1 
)

print("Modèle DeepSeek chargé avec succès.")

Chargement du modele deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Modele DeepSeek charge et pret.


In [ ]:
def classifier_avis_few_shot(avis_client):
    # Les avis Yelp sont en anglais, donc on donne des exemples en ANGLAIS.
    # C'est crucial pour que le modele reconnaisse le vocabulaire (pattern matching).
    # On garde les instructions en francais, DeepSeek comprend tres bien le melange.
    
    prompt = f"""Tu es un expert en analyse de sentiments.
Tache : Classe l'avis ci-dessous en POSITIF ou NEGATIF.
Reponds uniquement par le mot final.

Exemple 1 (Cuisine):
Avis: "Hands down the best pizza in Vegas! The crust was perfect."
Sentiment: POSITIF

Exemple 2 (Service):
Avis: "We waited 45 minutes for a table. The staff was rude."
Sentiment: NEGATIF

Exemple 3 (Ambiance):
Avis: "Super cute spot! The music was great."
Sentiment: POSITIF

Exemple 4 (Prix/Qualite):
Avis: "Bland food and way overpriced. I will not be coming back."
Sentiment: NEGATIF

--- A ton tour ---
Avis a classer : "{avis_client}"
Sentiment:"""

    # Generation de la reponse
    # On laisse 500 tokens pour que le modele puisse "reflechir" (<think>...)
    resultat = generator(
        prompt, 
        max_new_tokens=500, 
        do_sample=False,   # Resultat deterministe (stable)
        return_full_text=False, 
        pad_token_id=generator.tokenizer.eos_token_id
    )
    
    reponse_brute = resultat[0]['generated_text']
    
    # --- Nettoyage de la reponse ---
    reponse_upper = reponse_brute.upper()
    
    # On cherche les mots cles anglais (POSITIVE/NEGATIVE) ou francais (POSITIF/NEGATIF)
    if "NEGATIF" in reponse_upper or "NEGATIVE" in reponse_upper:
        return "NEGATIF"
    elif "POSITIF" in reponse_upper or "POSITIVE" in reponse_upper:
        return "POSITIF"
    else:
        # Si le modele a bavarde, on essaie de recuperer la fin apres sa reflexion
        return reponse_brute.split("</think>")[-1].strip()

# Test avec un vrai avis en anglais (Pilege : Food good / Service terrible)
print("--- Test DeepSeek Few-Shot (English) ---")
avis_test = "The food was actually quite good but the service was terrible, we waited forever."
print(f"Avis : {avis_test}")

res = classifier_avis_few_shot(avis_test)
print(f"Resultat du modele : {res}")

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Test DeepSeek Corrigé ---
Avis : La nourriture est bonne mais le service est catastrophique.
Resultat brut : Pour analyser le sentiment de l'avis "La nourriture est bonne mais le service est catastrophique", suivons les étapes suivantes :

1 **Identifie les termes pertinent pour le sentiment :**
   - "Nourriture" : positive
   - "Catastrophique" : negative

2 **Analyse de chaque terme :**
   - "La nourriture est bonne" : positive
   - "Le service est catastrophique" : negative

3 **Combinaux les termes pour obtenir le sentiment total :**
   - Le sentiment principal est "negative" car le service est catastrophique, ce qui est un sentiment negatif

**Réponse finale :** **NEGATIF**
